# 🫀 rPPG-Toolbox: DeepPhys Training & Evaluation on MMPD Dataset

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/drozanes/rPPG-Toolbox/blob/main/MMPD_DeepPhys_Colab.ipynb)

This notebook runs **DeepPhys** training and evaluation end-to-end on the complete **MMPD** benchmark dataset (all 33 subjects, 660 raw video files) using an NVIDIA T4 GPU on Google Colab.

### Key Engineering Highlights:
- **Zero Preprocessing Disk Footprint:** Uses Online Memory Streaming (`MMPDLoader` with LRU caching) to completely bypass the ~940 GB disk requirement.
- **Gated Hugging Face Token Included:** Seamless download of the 660 `.mat` files directly from `WeiQian98/mini_MMPD`.
- **CUDA Optimization & Contiguous Layout:** Fast training on Tesla T4 (~6.5 minutes per epoch).
- **Validation Tracking & Early Stopping:** Tracks validation loss each epoch (`patience=3`) and stops automatically upon convergence.
- **Automated Benchmark Recording:** Best checkpoint is automatically evaluated on the test set (MAE, RMSE, MAPE, Pearson $r$, SNR) and recorded into `rPPG_results_database.csv`.

In [ ]:
# 1. Verify GPU Allocation (Ensure runtime is GPU: Runtime -> Change runtime type -> T4 GPU)
!nvidia-smi

In [ ]:
# 2. Clone the Repository and Install Dependencies
!git clone https://github.com/drozanes/rPPG-Toolbox.git
%cd rPPG-Toolbox

# Install required dependencies (yacs, huggingface_hub, etc.)
!pip install -q yacs huggingface_hub pyyaml scipy opencv-python tqdm pandas matplotlib scikit-learn

In [ ]:
# 3. Disk Space Verification & Proactive Cleanup
import shutil
total, used, free = shutil.disk_usage("/")
print(f"Colab Root Disk: Total = {total/(1024**3):.1f} GB, Usable Free = {free/(1024**3):.1f} GB")

# If free space is below 55 GB, clear package caches for extra headroom
if free / (1024**3) < 55:
    print("Cleaning package cache for extra disk headroom...")
    !apt-get clean > /dev/null 2>&1
    !rm -rf /root/.cache/pip
    _, _, free = shutil.disk_usage("/")
    print(f"Updated Free Disk: {free/(1024**3):.1f} GB")

In [ ]:
# 4. Download Raw MMPD Mini Dataset (~47.3 GB, 660 files) from Hugging Face
# Uses token authentication for the gated WeiQian98/mini_MMPD repository
import os, glob, getpass
from huggingface_hub import snapshot_download

try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
except Exception:
    HF_TOKEN = None

if not HF_TOKEN:
    print("Please enter your Hugging Face Access Token (read access to WeiQian98/mini_MMPD):")
    HF_TOKEN = getpass.getpass("HF Token: ").strip()

REPO_ID = "WeiQian98/mini_MMPD"
data_dir = "./MMPD_dataset"
os.makedirs(data_dir, exist_ok=True)

print(f"Downloading all 660 .mat files from {REPO_ID} (takes ~12-15 minutes)...")
snapshot_download(
    repo_id=REPO_ID,
    repo_type="dataset",
    local_dir=data_dir,
    token=HF_TOKEN,
    ignore_patterns=[".DS_Store"],
    resume_download=True
)

mat_files = glob.glob(f"{data_dir}/**/*.mat", recursive=True)
print(f"✅ Download verification: {len(mat_files)} / 660 .mat files successfully downloaded!")
assert len(mat_files) == 660, f"Expected 660 files, found {len(mat_files)}"


In [ ]:
# 5. Launch DeepPhys Online Training with Validation Tracking & Early Stopping
# Training split:   Subjects 1-26 (5,200 chunks)
# Validation split: Subjects 27-29 (600 chunks)
# Testing split:    Subjects 30-33 (800 chunks)
# Speed on T4 GPU:  ~6.5 minutes per epoch (stops automatically upon convergence)

!python main.py --config_file configs/train_configs/MMPD_DEEPPHYS_ONLINE.yaml --device cuda --num_workers 2

In [ ]:
# 6. Display Updated Benchmark Results
import os, pandas as pd
if os.path.exists("rPPG_results_database.csv"):
    df = pd.read_csv("rPPG_results_database.csv")
    print("\n=== Benchmark Database (MMPD Entries) ===")
    display(df[df["Dataset"] == "MMPD"])
else:
    print("Results database not found.")

In [ ]:
# 7. Bundle Trained Weights, Checkpoints, and Benchmark Database
!zip -q -r mmpd_deepphys_trained_model.zip PreTrainedModels/ runs/ rPPG_results_database.csv
from google.colab import files
files.download("mmpd_deepphys_trained_model.zip")
print("\u2705 Download initiated for mmpd_deepphys_trained_model.zip!")